# Pose Estimation with AlphaPose + Retail Intelligence

**Pose estimation** finds the body keypoints (nose, shoulders, elbows, wrists, hips, knees, ankles, ...) of every person in an image or video. Compared to a simple bounding box, a skeleton tells us *where a person stands*, *how the body is oriented* and *what the arms are doing*: very useful information for business analytics.

In this notebook we use [AlphaPose](https://github.com/MVIG-SJTU/AlphaPose) (Shanghai Jiao Tong University), a **top-down** multi-person pose estimator:

| Step | Model | What it does |
|---|---|---|
| 1. Person detection | **YOLOv3-SPP** | finds a bounding box around every person in the frame |
| 2. Single person pose estimation (SPPE) | **SE-ResNet + DUC** (`duc_se.pth`) | for each box, predicts 17 heatmaps (one per keypoint) |
| 3. Parametric pose NMS | `pPose_nms.py` | removes duplicated skeletons of the same person |

The output for each person is **17 COCO keypoints**, each with `(x, y, confidence)`.

### What we will do
1. Check the GPU
2. Get the AlphaPose code
3. Install the few missing dependencies
4. Patch the legacy code so it runs with a modern PyTorch / Python
5. Download the pretrained models
6. Get an input video
7. Run AlphaPose on the video and watch the result
8. Understand the output (keypoints JSON)
9. **Retail intelligence**: turn skeletons into store KPIs (footfall, zone occupancy, dwell time, engagement, heatmaps, dashboard video)

> **Runtime:** this notebook needs a GPU. In Colab go to `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

### Why this notebook changed

The first version of this notebook tried to recreate the 2018 environment of AlphaPose inside Colab. Those steps no longer work on the current Colab runtime (Ubuntu 24.04, Python 3.13, PyTorch 2.x):

| Original step | Why it was there | What we do now |
|---|---|---|
| `apt-get install build-essential libssl-dev ...` | build tools to compile Python from source | not needed |
| `apt-get install python3.9` + `update-alternatives` | old PyTorch 1.12 wheels only exist for Python ≤ 3.10 | not needed: this changed only the shell `python`, **not** the notebook kernel, and Python 3.9 is not available on Ubuntu 24.04 |
| `pip install torch==1.12.1+cu113` | AlphaPose was written for old PyTorch versions | use the PyTorch already installed in Colab and **patch 3 lines** of legacy code |
| `pip install visdom` (failed to build) | imported by the pose model code | the import is unused, so we remove it |
| `pip install cython`, `apt-get install libyaml-dev` | needed by other AlphaPose branches | not needed by the `pytorch` branch |
| `yt-dlp` YouTube download | input video | still available, but YouTube often blocks downloads from Colab, so the default video now comes from the course GitHub repository |

## 0. Check the GPU

AlphaPose moves both networks to the GPU with `.cuda()`, so a GPU is mandatory. `nvidia-smi` shows the GPU model and memory, then we verify that PyTorch can use it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import torch
import torchvision

assert torch.cuda.is_available(), "No GPU found! In Colab go to Runtime -> Change runtime type -> T4 GPU"
print(f"PyTorch {torch.__version__} | torchvision {torchvision.__version__} | GPU: {torch.cuda.get_device_name(0)}")

## 1. Get the AlphaPose code

We clone the **`pytorch` branch** of AlphaPose (version 0.2). It is written in pure PyTorch, so, unlike the newer `master` branch, **it does not need to compile C++/CUDA extensions**.

`--depth 1` downloads only the latest commit (faster). The most important files are:

| File / folder | Role |
|---|---|
| `video_demo.py` | entry point: runs the whole pipeline on a video |
| `dataloader.py` | reads frames, runs YOLO, prepares person crops, writes the output video (in background threads) |
| `yolo/` | YOLOv3 person detector (Darknet config + code) |
| `SPPE/` | single person pose estimation network |
| `pPose_nms.py` | pose NMS + JSON writer |
| `fn.py` | drawing utilities (skeleton rendering) |
| `opt.py` | all command line options |

In [ ]:
import os

HOME = os.getcwd()  # /content in Google Colab
ALPHAPOSE_DIR = os.path.join(HOME, "AlphaPose")

!test -d "{ALPHAPOSE_DIR}" || git clone -q -b pytorch --depth 1 https://github.com/MVIG-SJTU/AlphaPose.git "{ALPHAPOSE_DIR}"
!ls "{ALPHAPOSE_DIR}"

## 2. Install the missing dependencies

Colab already provides PyTorch, torchvision, OpenCV, NumPy, SciPy, pandas, matplotlib and tqdm. We only need:
- `gdown`: downloads large files from Google Drive (the AlphaPose pretrained models are hosted there);
- `yt-dlp`: downloads YouTube videos (only for the optional YouTube input).

> We do **not** install AlphaPose's `requirements.txt`: it pins `torch==0.4.0`, which would break the Colab environment.

In [ ]:
!pip install -q gdown yt-dlp

## 3. Patch the legacy code

AlphaPose v0.2 was written in 2018. Three things it uses do not exist anymore:

| # | File | Old code | Problem | Fix |
|---|---|---|---|---|
| 1 | `fn.py` | `from torch._six import string_classes, int_classes` | `torch._six` (Python 2 compatibility) was removed in PyTorch 2.0 | use the built-in `str` and `int` |
| 2 | `fn.py` | `collections.Mapping`, `collections.Sequence` | moved to `collections.abc`, removed from `collections` in Python 3.10 | use `collections.abc` |
| 3 | `SPPE/src/main_fast_inference.py` | `import visdom` | `visdom` (a training dashboard) is imported but never used, and fails to install | remove the import |

The `patch_file` helper replaces the old code with the new one. It is safe to run it more than once.

In [ ]:
PATCHES = {
    "fn.py": [
        # 1. torch._six was removed in PyTorch 2.0
        ("from torch._six import string_classes, int_classes", "string_classes, int_classes = str, int"),
        # 2. collections.Mapping / collections.Sequence live in collections.abc since Python 3.10
        ("import collections\n", "import collections\nimport collections.abc\n"),
        ("collections.Mapping", "collections.abc.Mapping"),
        ("collections.Sequence", "collections.abc.Sequence"),
    ],
    "SPPE/src/main_fast_inference.py": [
        # 3. unused import of a package that is not installed
        ("import visdom\n", ""),
    ],
}


def patch_file(relative_path, replacements):
    path = os.path.join(ALPHAPOSE_DIR, relative_path)
    with open(path) as f:
        source = f.read()
    for old, new in replacements:
        if new and new in source:  # already patched
            continue
        if old not in source:
            if not new:  # line already removed
                continue
            raise ValueError(f"{relative_path}: code to patch not found: {old!r}")
        source = source.replace(old, new)
    with open(path, "w") as f:
        f.write(source)
    print("patched", relative_path)


for relative_path, replacements in PATCHES.items():
    patch_file(relative_path, replacements)

## 4. Download the pretrained models

AlphaPose needs two pretrained networks, both trained on the [COCO dataset](https://cocodataset.org/):

| File | Model | Size |
|---|---|---|
| `models/yolo/yolov3-spp.weights` | YOLOv3-SPP person detector (Darknet format) | 252 MB |
| `models/sppe/duc_se.pth` | SPPE pose network: SE-ResNet backbone + DUC (Dense Upsampling Convolution) head | 239 MB |

They must be saved exactly in these folders, because the AlphaPose code loads them with these relative paths. The YOLO weights come from the official YOLO website, with the AlphaPose Google Drive copy as a fallback. Already downloaded files are skipped.

In [ ]:
import urllib.request
import gdown


def download_model(relative_path, expected_mb, gdrive_id, url=None):
    path = os.path.join(ALPHAPOSE_DIR, relative_path)
    is_complete = lambda: os.path.exists(path) and os.path.getsize(path) > 0.95 * expected_mb * 1e6
    if is_complete():
        print(f"already downloaded: {relative_path}")
        return
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if url is not None:
        try:
            urllib.request.urlretrieve(url, path)
        except Exception as error:
            print(f"download from {url} failed ({error}): using Google Drive")
    if not is_complete():
        gdown.download(id=gdrive_id, output=path, quiet=False)
    assert is_complete(), f"{relative_path} is incomplete: run this cell again"
    print(f"{relative_path}: {os.path.getsize(path) / 1e6:.0f} MB")


# Single person pose estimator (SPPE)
download_model("models/sppe/duc_se.pth", 239, gdrive_id="1OPORTWB2cwd5YTVBX-NE8fsauZJWsrtW")
# Person detector
download_model("models/yolo/yolov3-spp.weights", 252, gdrive_id="1D47msNOOiJKvPOXlnpyzdKA3k6E97NTC",
               url="https://pjreddie.com/media/files/yolov3-spp.weights")

## 5. Get the input video

Two options:
- **`"github"` (default)**: a shopping mall video from the course GitHub repository. We also use it for the retail intelligence example.
- **`"youtube"`**: the original demo video ([David Armand - Human, interpretative dance](https://www.youtube.com/watch?v=0vnMdFmZ3P8)), downloaded with `yt-dlp`. YouTube often blocks downloads from Colab servers ("Sign in to confirm you're not a bot"); in that case use the GitHub video. The retail analysis in section 9 is designed for the mall video.

Then `ffmpeg` keeps only the first `CLIP_SECONDS` seconds (`-t`) and removes the audio track (`-an`), so AlphaPose processes a short clip.

In [ ]:
import subprocess
from IPython.display import YouTubeVideo, display

VIDEO_SOURCE = "github"  # "github" or "youtube"
CLIP_SECONDS = 14        # the mall video lasts ~14 s: we process all of it

if VIDEO_SOURCE == "github":
    VIDEO_URL = "https://raw.githubusercontent.com/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business/main/02-Pytorch%20and%20CV/03_tracking/media/mall_crowd.mp4"
    urllib.request.urlretrieve(VIDEO_URL, "input_full.mp4")
else:
    YOUTUBE_ID = "0vnMdFmZ3P8"
    display(YouTubeVideo(YOUTUBE_ID))
    subprocess.run(["yt-dlp", "--force-overwrites", "-f", "b[ext=mp4]/bv*+ba/b", "--merge-output-format", "mp4",
                    "-o", "input_full.mp4", f"https://www.youtube.com/watch?v={YOUTUBE_ID}"], check=True)

!ffmpeg -y -loglevel error -i input_full.mp4 -t {CLIP_SECONDS} -an video.mp4

Let's check the clip properties and look at the first frame.

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture("video.mp4")
FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
ok, first_frame = cap.read()
cap.release()
assert ok, "Cannot read video.mp4"

print(f"{W}x{H} pixels | {FPS:.0f} fps | {N_FRAMES} frames ({N_FRAMES / FPS:.1f} s)")
plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
plt.title("First frame")
plt.axis("off")
plt.show()

## 6. Run AlphaPose on the video

`video_demo.py` runs the whole pipeline frame by frame: **read frame → YOLO person boxes → crop each person → SPPE heatmaps → keypoints → pose NMS → draw + save**.

| Option | Meaning |
|---|---|
| `--sp` | single process mode: uses threads instead of multiple processes (more robust in Colab) |
| `--video` | input video |
| `--outdir` | output folder |
| `--save_video` | save the video with the skeletons drawn on it |
| `--vis_fast` | faster (simpler) skeleton rendering |
| `--conf`, `--nms` | (optional) detection confidence and NMS thresholds, defaults `0.05` and `0.6` |

We run the script from the `AlphaPose` folder because it loads the models with relative paths. It produces:
- `AlphaPose_video.avi`: the video with the skeletons;
- `alphapose-results.json`: all the keypoints (used in the next sections).

AlphaPose processes one person crop at a time, so crowded scenes are slow: with ~37 people per frame the 14 seconds clip takes **about 5-10 minutes** (≈ 7 minutes on a laptop RTX 4070 in our test).

In [ ]:
OUTPUT_DIR = os.path.join(HOME, "alphapose_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

!cd "{ALPHAPOSE_DIR}" && python video_demo.py --sp --video "{HOME}/video.mp4" --outdir "{OUTPUT_DIR}" --save_video --vis_fast
!ls -la "{OUTPUT_DIR}"

## 7. Watch the result

OpenCV saved an `.avi` file (XVID codec) that browsers cannot play. `ffmpeg` converts it to an `.mp4` with the H.264 codec (`-c:v libx264 -pix_fmt yuv420p`), then we embed it in the notebook as a base64 string.

In [ ]:
import base64
from IPython.display import HTML

!ffmpeg -y -loglevel error -i "{OUTPUT_DIR}/AlphaPose_video.avi" -c:v libx264 -pix_fmt yuv420p -crf 28 "{OUTPUT_DIR}/AlphaPose_video.mp4"


def show_local_mp4_video(file_name, width=960):
    with open(file_name, "rb") as f:
        video_encoded = base64.b64encode(f.read()).decode("ascii")
    return HTML(f'<video width="{width}" controls><source src="data:video/mp4;base64,{video_encoded}" type="video/mp4"></video>')


show_local_mp4_video(os.path.join(OUTPUT_DIR, "AlphaPose_video.mp4"))

## 8. Understand the output

`alphapose-results.json` is a list with **one element per detected person per frame** (COCO results format):

```json
{"image_id": "0.jpg", "category_id": 1, "keypoints": [x1, y1, c1, x2, y2, c2, ...], "score": 2.87}
```

- `image_id`: the frame index (`"0.jpg"` = first frame);
- `keypoints`: 17 × 3 numbers = `(x, y, confidence)` of each keypoint, in pixels;
- `score`: confidence of the whole pose.

The 17 COCO keypoints, in this order:

| Index | Keypoint | Index | Keypoint | Index | Keypoint |
|---|---|---|---|---|---|
| 0 | nose | 6 | right shoulder | 12 | right hip |
| 1 | left eye | 7 | left elbow | 13 | left knee |
| 2 | right eye | 8 | right elbow | 14 | right knee |
| 3 | left ear | 9 | left wrist | 15 | left ankle |
| 4 | right ear | 10 | right wrist | 16 | right ankle |
| 5 | left shoulder | 11 | left hip | | |

We load it into a pandas DataFrame with one row per pose and the keypoints as a `(17, 3)` NumPy array.

In [ ]:
import json
import numpy as np
import pandas as pd

KEYPOINT_NAMES = ["nose", "left_eye", "right_eye", "left_ear", "right_ear",
                  "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
                  "left_wrist", "right_wrist", "left_hip", "right_hip",
                  "left_knee", "right_knee", "left_ankle", "right_ankle"]
SKELETON = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
            (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]

with open(os.path.join(OUTPUT_DIR, "alphapose-results.json")) as f:
    results = json.load(f)
print(f"{len(results)} poses detected. First one:\n{results[0]}")

poses = pd.DataFrame({
    "frame": [int(os.path.splitext(r["image_id"])[0]) for r in results],
    "score": [r["score"] for r in results],
    "keypoints": [np.array(r["keypoints"]).reshape(17, 3) for r in results],
})
poses = poses.sort_values("frame", kind="stable").reset_index(drop=True)
print(f"\nframes with people: {poses['frame'].nunique()} | average people per frame: {poses.groupby('frame').size().mean():.1f}")
poses.head()

The keypoint confidence tells us how reliable each point is (occluded body parts get low values). Let's look at its distribution and choose a threshold `KP_THRESHOLD` to ignore unreliable keypoints, then draw the skeletons of one frame.

In [ ]:
all_conf = np.stack(poses["keypoints"])[:, :, 2]  # (n_poses, 17)
conf_by_keypoint = pd.Series(all_conf.mean(axis=0), index=KEYPOINT_NAMES)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].hist(all_conf.ravel(), bins=50)
axes[0].set_title("Keypoint confidence distribution")
conf_by_keypoint.plot.bar(ax=axes[1], title="Average confidence per keypoint")
plt.tight_layout()
plt.show()

KP_THRESHOLD = 0.4


def read_frame(frame_index, video_path="video.mp4"):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_index)
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


def plot_skeletons(frame_index):
    plt.figure(figsize=(14, 8))
    plt.imshow(read_frame(frame_index))
    for kp in poses.loc[poses["frame"] == frame_index, "keypoints"]:
        visible = kp[:, 2] > KP_THRESHOLD
        for a, b in SKELETON:
            if visible[a] and visible[b]:
                plt.plot([kp[a, 0], kp[b, 0]], [kp[a, 1], kp[b, 1]], "c-", linewidth=2)
        plt.scatter(kp[visible, 0], kp[visible, 1], c="yellow", s=12, zorder=3)
    plt.title(f"Frame {frame_index}: keypoints with confidence > {KP_THRESHOLD}")
    plt.axis("off")
    plt.show()


plot_skeletons(frame_index=0)

# 9. Retail intelligence with pose estimation

Retailers and shopping malls want to understand **how people use the space** to design layouts, place promotions and staff the store. With a camera and pose estimation we can measure:

| KPI | Business question | How we get it from skeletons |
|---|---|---|
| **Footfall** | How many visitors? | count the people we follow over time |
| **Zone occupancy** | Which areas are crowded, and when? | people whose feet are inside a zone, frame by frame |
| **Dwell time** | How long do visitors stay in each area? | time spent by each visitor inside each zone |
| **Engagement** | Do people stop at the promo stand, or just walk by? | visitors standing still in the zone |
| **Interactions** | Do people reach for products / raise their arms? | wrist above the shoulder |
| **Heatmap** | Where do people walk and stand? | density of floor positions |

**Why skeletons instead of boxes?** The ankles give the exact point where a person *touches the floor* (a box is also enlarged by bags and arms), and the arms tell us about gestures. Skeletons also contain no face or identity information, which helps with privacy (GDPR).

The analysis pipeline:
1. **Features**: floor position and arm gestures of each pose.
2. **Tracking**: link the poses of consecutive frames to follow each visitor.
3. **Zones**: define the store areas.
4. **KPIs**: occupancy, dwell time, engagement, interactions.
5. **Heatmap** and **dashboard video**.

## 9.1 From keypoints to features

For each pose we compute:
- **floor point**: the midpoint of the visible ankles. If both ankles are occluded, we use the lowest visible keypoint;
- **body height** (pixels): from the highest to the lowest visible keypoint, useful to understand the scale of each person;
- **arm raised**: at least one wrist *above* its shoulder (in images `y` grows downwards, so "above" means a smaller `y`). In a store this is a proxy for reaching a shelf, taking a product or pointing.

In [ ]:
L_SHOULDER, R_SHOULDER, L_WRIST, R_WRIST, L_ANKLE, R_ANKLE = 5, 6, 9, 10, 15, 16


def pose_features(kp, threshold=KP_THRESHOLD):
    visible = kp[:, 2] > threshold
    if visible.sum() < 5:  # too few reliable keypoints: skip this pose
        return None
    ankles = [kp[i, :2] for i in (L_ANKLE, R_ANKLE) if visible[i]]
    floor = np.mean(ankles, axis=0) if ankles else kp[visible][np.argmax(kp[visible, 1]), :2]
    arm_raised = any(visible[w] and visible[s] and kp[w, 1] < kp[s, 1]
                     for w, s in ((L_WRIST, L_SHOULDER), (R_WRIST, R_SHOULDER)))
    return {"floor_x": floor[0], "floor_y": floor[1],
            "body_height": kp[visible, 1].max() - kp[visible, 1].min(),
            "arm_raised": arm_raised}


features = pd.DataFrame([pose_features(kp) or {} for kp in poses["keypoints"]], index=poses.index)
people = pd.concat([poses, features], axis=1).dropna(subset=["floor_x"]).reset_index(drop=True)
people["arm_raised"] = people["arm_raised"].astype(bool)
people["time_s"] = people["frame"] / FPS

print(f"{len(people)} valid poses out of {len(poses)} | median body height: {people['body_height'].median():.0f} px")
people[["frame", "time_s", "floor_x", "floor_y", "body_height", "arm_raised"]].head()

## 9.2 Tracking: following each visitor over time

AlphaPose gives us independent skeletons in every frame: it does not know that the person in frame 10 is the same as in frame 11. To compute dwell times we need a **visitor ID**.

We use a simple tracker based on the **floor points**:
1. for each new frame, compute the distance between every new floor point and the last position of every active track;
2. find the best one-to-one assignment with the **Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`);
3. accept a match only if the distance is small: at most `MAX_STEP_PX` per frame since the track was last seen (and never more than `MAX_DISTANCE_PX`), so a lost track cannot jump onto another person; unmatched points start new tracks;
4. a track that is not seen for more than `MAX_MISSING_FRAMES` frames is closed.

Finally we remove very short tracks (less than `MIN_TRACK_SECONDS`), which are usually false detections.

> AlphaPose also offers a dedicated pose tracker ([PoseFlow](https://github.com/MVIG-SJTU/AlphaPose/tree/pytorch/PoseFlow)); this simple version is enough for our analytics and easy to understand.

In [ ]:
from scipy.optimize import linear_sum_assignment

MAX_STEP_PX = 20          # max movement of the floor point in one frame
MAX_DISTANCE_PX = 40      # max movement after a few missed frames
MAX_MISSING_FRAMES = 10   # keep a lost track alive for this many frames (occlusions)
MIN_TRACK_SECONDS = 1.0   # drop tracks shorter than this


def track_people(df):
    track_ids = np.full(len(df), -1)
    tracks = {}  # track_id -> (last_position, last_frame)
    next_id = 0
    for frame, group in df.groupby("frame"):
        positions = group[["floor_x", "floor_y"]].to_numpy()
        active = [t for t, (_, last_frame) in tracks.items() if frame - last_frame <= MAX_MISSING_FRAMES]
        assigned = {}
        if active:
            last_positions = np.array([tracks[t][0] for t in active])
            frames_gap = np.array([frame - tracks[t][1] for t in active])
            max_distance = np.minimum(MAX_STEP_PX * frames_gap, MAX_DISTANCE_PX)
            cost = np.linalg.norm(positions[:, None, :] - last_positions[None, :, :], axis=2)
            for row, col in zip(*linear_sum_assignment(cost)):
                if cost[row, col] <= max_distance[col]:
                    assigned[row] = active[col]
        for row, index in enumerate(group.index):
            if row not in assigned:
                assigned[row] = next_id
                next_id += 1
            track_ids[index] = assigned[row]
            tracks[assigned[row]] = (positions[row], frame)
    return track_ids


people["track_id"] = track_people(people)
track_duration = people.groupby("track_id")["frame"].agg(lambda f: (f.max() - f.min() + 1) / FPS)
valid_tracks = track_duration[track_duration >= MIN_TRACK_SECONDS].index
people = people[people["track_id"].isin(valid_tracks)].reset_index(drop=True)

print(f"visitors tracked: {people['track_id'].nunique()} (removed {len(track_duration) - len(valid_tracks)} short tracks)")

Now we can compute the **walking speed** of each visitor: the displacement of the floor point over the last second.

**Perspective problem:** people far from the camera (top of the frame) look smaller, so they move fewer pixels per second even when they walk at the same speed. We therefore measure speed in **body heights per second** (the pixel speed divided by the median body height of that visitor): a walking person covers about 0.5-1 body heights per second, whatever their distance from the camera.

Visitors with a speed below `STATIONARY_SPEED` are **standing still** (browsing, talking, waiting), the others are **walking**.

In [ ]:
STATIONARY_SPEED = 0.2  # body heights per second

people = people.sort_values(["track_id", "frame"]).reset_index(drop=True)
window = max(1, int(round(FPS)))  # frames in one second
grouped = people.groupby("track_id")
dx = people["floor_x"] - grouped["floor_x"].shift(window)
dy = people["floor_y"] - grouped["floor_y"].shift(window)
dt = (people["frame"] - grouped["frame"].shift(window)) / FPS
people["speed_px_s"] = np.sqrt(dx ** 2 + dy ** 2) / dt
people["speed_px_s"] = people.groupby("track_id")["speed_px_s"].transform(lambda s: s.bfill())
people["speed"] = people["speed_px_s"] / people.groupby("track_id")["body_height"].transform("median")
people["stationary"] = people["speed"] < STATIONARY_SPEED

print(f"median speed: {people['speed_px_s'].median():.0f} px/s = {people['speed'].median():.2f} body heights/s | poses standing still: {people['stationary'].mean():.0%}")

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB), alpha=0.6)
for track_id, track in people.groupby("track_id"):
    plt.plot(track["floor_x"], track["floor_y"], linewidth=2)
    plt.text(track["floor_x"].iloc[-1], track["floor_y"].iloc[-1], str(track_id), fontsize=8, color="red")
plt.title("Visitor trajectories (floor points)")
plt.axis("off")
plt.show()

## 9.3 Store zones

In a real project the zones come from the store layout (entrance, shelves, promo stands, checkout) and are drawn once on the camera view. Here we define them as rectangles in **normalized coordinates** (`0-1` of the frame width and height), so they do not depend on the video resolution.

A person belongs to a zone if their **floor point** is inside it. The zones are checked in order, and the first match wins.

In [ ]:
ZONES = {  # name: (x_min, y_min, x_max, y_max), normalized coordinates
    "Promo stand": (0.12, 0.38, 0.33, 0.68),
    "Entrance": (0.00, 0.00, 1.00, 0.18),
    "Left corridor": (0.00, 0.18, 0.33, 1.00),
    "Main aisle": (0.33, 0.18, 0.70, 1.00),
    "Showcase": (0.70, 0.18, 1.00, 1.00),
}
ZONE_COLORS = {name: color for name, color in zip(ZONES, plt.cm.Set1.colors)}


def zone_of(x, y):
    for name, (x0, y0, x1, y1) in ZONES.items():
        if x0 * W <= x < x1 * W and y0 * H <= y < y1 * H:
            return name
    return "Other"


people["zone"] = [zone_of(x, y) for x, y in zip(people["floor_x"], people["floor_y"])]

plt.figure(figsize=(14, 8))
plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
for name, (x0, y0, x1, y1) in reversed(list(ZONES.items())):
    plt.gca().add_patch(plt.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H, alpha=0.3, color=ZONE_COLORS[name]))
    plt.text((x0 + 0.01) * W, (y0 + 0.04) * H, name, fontsize=13, weight="bold", color="black")
plt.title("Store zones")
plt.axis("off")
plt.show()

## 9.4 Retail KPIs

**Occupancy over time**: how many people are in each zone at every moment. Peaks show crowding (e.g. to open another checkout or send staff to an area).

In [ ]:
occupancy = people.pivot_table(index="frame", columns="zone", values="track_id", aggfunc="count", fill_value=0)
occupancy = occupancy.reindex(range(N_FRAMES), fill_value=0)
occupancy.index = occupancy.index / FPS

ax = occupancy.rolling(int(FPS), min_periods=1).mean().plot(figsize=(14, 5), color=[ZONE_COLORS.get(z, "gray") for z in occupancy.columns])
ax.set_xlabel("time (s)")
ax.set_ylabel("people in zone (1 s moving average)")
ax.set_title("Zone occupancy over time")
plt.show()

print("Peak occupancy per zone:")
print(occupancy.max().sort_values(ascending=False).to_string())

**Zone report**: for each zone we compute
- **visitors**: unique visitors that entered the zone;
- **avg dwell time**: average seconds a visitor spent in the zone;
- **engagement rate**: share of visitors who **stood still for at least `ENGAGED_SECONDS`** in the zone (they did not just walk by). For a promo stand this is the key metric: how many passers-by stopped;
- **arm raises**: share of time with a raised arm (proxy for product interactions).

In [ ]:
ENGAGED_SECONDS = 1.0

visits = people.groupby(["zone", "track_id"]).agg(
    dwell_s=("frame", lambda f: len(f) / FPS),
    still_s=("stationary", lambda s: s.sum() / FPS),
    arm_raised_s=("arm_raised", lambda a: a.sum() / FPS),
).reset_index()
visits["engaged"] = visits["still_s"] >= ENGAGED_SECONDS

zone_report = visits.groupby("zone").agg(
    visitors=("track_id", "nunique"),
    avg_dwell_s=("dwell_s", "mean"),
    total_person_s=("dwell_s", "sum"),
    engagement_rate=("engaged", "mean"),
    arm_raised_share=("arm_raised_s", "sum"),
)
zone_report["arm_raised_share"] = zone_report["arm_raised_share"] / zone_report["total_person_s"]
zone_report = zone_report.sort_values("total_person_s", ascending=False)

print(f"Total footfall (unique visitors): {people['track_id'].nunique()}")
display(zone_report.style.format({"avg_dwell_s": "{:.1f}", "total_person_s": "{:.1f}",
                                  "engagement_rate": "{:.0%}", "arm_raised_share": "{:.0%}"}))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
colors = [ZONE_COLORS.get(z, "gray") for z in zone_report.index]
zone_report["visitors"].plot.bar(ax=axes[0], color=colors, title="Visitors per zone")
zone_report["avg_dwell_s"].plot.bar(ax=axes[1], color=colors, title="Average dwell time (s)")
zone_report["engagement_rate"].plot.bar(ax=axes[2], color=colors, title=f"Engagement rate (still >= {ENGAGED_SECONDS:.0f} s)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 9.5 Heatmap of the store

A **heatmap** of the floor points shows the most used paths and the areas where people stop. We accumulate all floor points in a 2D histogram, smooth it with a Gaussian blur and overlay it on the frame. The color scale is clipped at the 99th percentile, otherwise a few very crowded spots would hide the walking paths.

We draw two maps: all positions (**traffic**) and only people standing still (**attention areas**).

In [ ]:
def floor_heatmap(df, cell_px=10, blur_px=41):
    hist, _, _ = np.histogram2d(df["floor_y"], df["floor_x"], bins=(H // cell_px, W // cell_px), range=[[0, H], [0, W]])
    heat = cv2.resize(hist.astype(np.float32), (W, H), interpolation=cv2.INTER_LINEAR)
    heat = cv2.GaussianBlur(heat, (blur_px, blur_px), 0)
    top = np.percentile(heat[heat > 0], 99) if (heat > 0).any() else 1
    return np.clip(heat / top, 0, 1)


fig, axes = plt.subplots(1, 2, figsize=(20, 6))
background = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)
for ax, (title, subset) in zip(axes, [("Traffic: all positions", people),
                                      ("Attention areas: people standing still", people[people["stationary"]])]):
    ax.imshow(background)
    ax.imshow(floor_heatmap(subset), cmap="jet", alpha=0.5)
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 9.6 Retail dashboard video

Finally we put everything together in a video for store managers: zones with their **live occupancy**, each visitor with **ID**, **trajectory** and **time spent in the current zone**. Visitors standing still are drawn in **red**, walking visitors in **green**, and a raised arm is marked with **"ARM"**.

In [ ]:
DASHBOARD_PATH = os.path.join(OUTPUT_DIR, "retail_dashboard.mp4")
TRAIL_FRAMES = int(2 * FPS)

people["zone_time_s"] = people.groupby(["track_id", "zone"]).cumcount() / FPS

cap = cv2.VideoCapture("video.mp4")
writer = cv2.VideoWriter(os.path.join(OUTPUT_DIR, "retail_dashboard.avi"), cv2.VideoWriter_fourcc(*"MJPG"), FPS, (W, H))
frame_index = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    overlay = frame.copy()
    current = people[people["frame"] == frame_index]
    counts = current["zone"].value_counts()
    for name, (x0, y0, x1, y1) in reversed(list(ZONES.items())):
        color = tuple(int(255 * c) for c in ZONE_COLORS[name][::-1])  # RGB (0-1) -> BGR (0-255)
        cv2.rectangle(overlay, (int(x0 * W), int(y0 * H)), (int(x1 * W), int(y1 * H)), color, -1)
    frame = cv2.addWeighted(overlay, 0.25, frame, 0.75, 0)
    for name, (x0, y0, x1, y1) in ZONES.items():
        cv2.putText(frame, f"{name}: {counts.get(name, 0)}", (int(x0 * W) + 8, int(y0 * H) + 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 2)

    trail = people[(people["frame"] > frame_index - TRAIL_FRAMES) & (people["frame"] <= frame_index)]
    for _, row in current.iterrows():
        color = (0, 0, 255) if row["stationary"] else (0, 200, 0)
        points = trail.loc[trail["track_id"] == row["track_id"], ["floor_x", "floor_y"]].to_numpy(np.int32)
        cv2.polylines(frame, [points], False, color, 2)
        center = (int(row["floor_x"]), int(row["floor_y"]))
        cv2.circle(frame, center, 6, color, -1)
        label = f"#{row['track_id']} {row['zone_time_s']:.0f}s" + (" ARM" if row["arm_raised"] else "")
        cv2.putText(frame, label, (center[0] + 8, center[1] + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    cv2.putText(frame, f"t={frame_index / FPS:4.1f}s  people={len(current)}  visitors so far={people.loc[people['frame'] <= frame_index, 'track_id'].nunique()}",
                (10, H - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    writer.write(frame)
    frame_index += 1
cap.release()
writer.release()

!ffmpeg -y -loglevel error -i "{OUTPUT_DIR}/retail_dashboard.avi" -c:v libx264 -pix_fmt yuv420p -crf 28 "{DASHBOARD_PATH}"
show_local_mp4_video(DASHBOARD_PATH)

## 10. Discussion and next steps

**What we built:** from a plain camera video to store KPIs (footfall, occupancy, dwell time, engagement, interactions, heatmaps) using only body keypoints.

**Limitations of this demo**
- **Camera perspective**: distances are in pixels. With a calibration (a *homography* from 4 known floor points) floor points can be converted to meters, so speeds and areas become real measures.
- **Tracking**: our tracker only uses positions; crossing people can swap IDs. Dedicated trackers (PoseFlow, ByteTrack, DeepSORT) also use appearance.
- **Occlusions and crowds**: hidden ankles make the floor point less accurate.
- **Thresholds** (`KP_THRESHOLD`, `STATIONARY_SPEED`, `ENGAGED_SECONDS`, zones) must be tuned for each camera and store.
- **Privacy**: even if skeletons are anonymous, video analytics in stores requires a legal basis and clear information to customers (GDPR).

**Modern alternatives**: AlphaPose 0.6 (136 whole-body keypoints, needs compiled extensions), Ultralytics `yolo11n-pose` / `yolo26n-pose` (detection + pose + tracking in one model), RTMPose (MMPose).

**Exercises**
1. Move the `Promo stand` zone to another area: how do visitors, dwell time and engagement change?
2. Add a "queue" alert: raise a warning when a zone has more than N people for more than 5 seconds.
3. Compute the direction of movement (entering vs leaving) of the people crossing the `Entrance` zone.
4. Replace our tracker with Ultralytics tracking on the same video (see the tracking notebooks of the course) and compare the number of visitors.